# Component 3 — Data Audit 01

Read-only audit of the fisheries workbooks, chlorophyll-a NetCDF file, and SST/SSS NetCDF file. This notebook does not clean, transform, overwrite, or export raw data, and it does not train a model.

In [5]:
from __future__ import annotations
import json, os, re, sys
from datetime import datetime, timedelta
from pathlib import Path
from typing import Any
import h5py
import numpy as np
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display = print
from openpyxl import load_workbook
pd.set_option('display.max_columns', 60)
pd.set_option('display.max_colwidth', 100)
FISHERIES_SAMPLE_ROWS = 10_000
NULL_TOKENS = {'', 'null', 'none', 'na', 'n/a', 'nan', '-'}


## 1. Configure Google Colab or local Windows paths

In Google Colab, this cell mounts Google Drive and uses `MyDrive/FishGuard-C3`. Locally, it uses the external raw-data folder beside `Research Project main`. Raw files are opened only for reading.

In [6]:
def find_component_root(start: Path) -> Path:
    anchors = [start.resolve(), *start.resolve().parents]
    candidates = []
    for anchor in anchors:
        candidates.extend([
            anchor,
            anchor / 'component - 3',
            anchor / 'J26-DS-350' / 'component - 3',
        ])
    for candidate in candidates:
        if (candidate / 'Notebooks').is_dir() and (candidate / 'src').is_dir():
            return candidate.resolve()
    raise FileNotFoundError(
        f'Could not locate the repository Component 3 root from {start.resolve()}. '
        'Open the J26-DS-350 repository or set the notebook working directory inside it.'
    )

RUNNING_IN_COLAB = 'google.colab' in sys.modules
if RUNNING_IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive/FishGuard-C3')
    COMPONENT_ROOT = None
    RAW_ROOT = DRIVE_ROOT / 'data' / 'raw'
    AUDIT_OUTPUT_DIR = DRIVE_ROOT / 'outputs' / 'audit'
else:
    COMPONENT_ROOT = find_component_root(Path.cwd())
    configured = os.getenv('FISHGUARD_C3_RAW_DATA')
    RAW_ROOT = Path(configured).expanduser().resolve() if configured else COMPONENT_ROOT.parents[2] / 'component - 3' / 'data' / 'Raw'
    configured_output = os.getenv('FISHGUARD_C3_AUDIT_OUTPUT')
    AUDIT_OUTPUT_DIR = Path(configured_output).expanduser().resolve() if configured_output else COMPONENT_ROOT / 'outputs' / 'audit'
FISHERIES_DIR, CHL_DIR, SST_DIR = RAW_ROOT / 'Fisheries', RAW_ROOT / 'chlorophyll_a', RAW_ROOT / 'sst'
missing = [path for path in [RAW_ROOT, FISHERIES_DIR, CHL_DIR, SST_DIR] if not path.is_dir()]
if missing: raise FileNotFoundError(f'Missing required directories: {missing}')
AUDIT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Runtime: {"Google Colab" if RUNNING_IN_COLAB else "Local Windows"}')
print(f'Component root: {COMPONENT_ROOT or "not required in Colab"}')
print(f'External raw-data root: {RAW_ROOT}')
print(f'Audit output directory: {AUDIT_OUTPUT_DIR}')
print('Raw files will only be opened for reading.')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Runtime: Google Colab
Component root: not required in Colab
External raw-data root: /content/drive/MyDrive/FishGuard-C3/data/raw
Audit output directory: /content/drive/MyDrive/FishGuard-C3/outputs/audit
Raw files will only be opened for reading.


## 2. Dataset inventory

In [7]:
inventory = pd.DataFrame([
    {'dataset_group': p.parent.name, 'file_name': p.name, 'extension': p.suffix.lower(),
     'size_mb': round(p.stat().st_size / 1024**2, 2), 'relative_path': str(p.relative_to(RAW_ROOT))}
    for p in sorted(RAW_ROOT.rglob('*')) if p.is_file()
])
display(inventory)
fisheries_files = sorted(FISHERIES_DIR.glob('*.xlsx'))
chl_files, sst_files = sorted(CHL_DIR.glob('*.nc')), sorted(SST_DIR.glob('*.nc'))
assert fisheries_files and chl_files and sst_files, 'One or more required dataset groups are empty.'


,dataset_group,file_name,extension,size_mb,relative_path
0,Fisheries,2017 Log Data.xlsx,.xlsx,81.18,Fisheries/2017 Log Data.xlsx
1,Fisheries,2018 Log data.xlsx,.xlsx,62.74,Fisheries/2018 Log data.xlsx
2,Fisheries,2019 Log data.xlsx,.xlsx,58.59,Fisheries/2019 Log data.xlsx
3,Fisheries,2020 Log Data.xlsx,.xlsx,20.77,Fisheries/2020 Log Data.xlsx
4,Fisheries,2021 LOGBOOK data final.xlsx,.xlsx,58.56,Fisheries/2021 LOGBOOK data final.xlsx
5,Fisheries,2022 Log Data.xlsx,.xlsx,13.49,Fisheries/2022 Log Data.xlsx
6,Fisheries,2023 Log Data.xlsx,.xlsx,18.20,Fisheries/2023 Log Data.xlsx
7,Fisheries,2024 Log Data.xlsx,.xlsx,16.05,Fisheries/2024 Log Data.xlsx
8,chlorophyll_a,Chla_Western_Southern_SriLanka_daily.nc,.nc,49.80,chlorophyll_a/Chla_Western_Southern_SriLanka_daily.nc
9,sst,SST_SSS_Western_Southern_SriLanka_daily.nc,.nc,13.96,sst/SST_SSS_Western_Southern_SriLanka_daily.nc


## 3. Fisheries workbook structure

Excel-reported row counts are inventory values, not yet verified non-empty observation counts. Similar field names are not assumed to share definitions or units.

In [8]:
def clean_header(value: Any, position: int) -> str:
    if value is None or not str(value).strip(): return f'<blank_{position}>'
    return re.sub(r'\s+', ' ', str(value).strip())

def inspect_workbook(path: Path):
    workbook = load_workbook(path, read_only=True, data_only=False)
    records, headers = [], {}
    try:
        for sheet in workbook.worksheets:
            first = []
            for row in sheet.iter_rows(min_row=1, max_row=min(20, sheet.max_row), values_only=True):
                if any(v is not None for v in row): first = list(row); break
            names = [clean_header(v, i + 1) for i, v in enumerate(first)]
            key = f'{path.name} :: {sheet.title}'
            headers[key] = names
            records.append({'file_name': path.name, 'sheet_name': sheet.title,
                'reported_rows': sheet.max_row, 'reported_columns': sheet.max_column,
                'header_fields': len(names), 'blank_headers': sum(n.startswith('<blank_') for n in names),
                'duplicate_headers': sorted({n for n in names if names.count(n) > 1})})
    finally: workbook.close()
    return records, headers

structure_records, headers_by_sheet = [], {}
for path in fisheries_files:
    records, headers = inspect_workbook(path)
    structure_records.extend(records); headers_by_sheet.update(headers)
fisheries_structure = pd.DataFrame(structure_records)
display(fisheries_structure)
for key, fields in headers_by_sheet.items(): print(f'\n{key}\n{fields}')


,file_name,sheet_name,reported_rows,reported_columns,header_fields,blank_headers,duplicate_headers
0,2017 Log Data.xlsx,Sheet1,604312,29,29,15,[]
1,2018 Log data.xlsx,2018 Log data,363235,39,39,0,[]
2,2019 Log data.xlsx,Sheet1,408748,35,35,4,[]
3,2020 Log Data.xlsx,Sheet5,284627,15,15,0,[]
4,2021 LOGBOOK data final.xlsx,trip 2021,280620,41,41,0,[id]
5,2021 LOGBOOK data final.xlsx,summery,280620,17,17,13,[]
6,2021 LOGBOOK data final.xlsx,Sheet3,1,1,0,0,[]
7,2022 Log Data.xlsx,catch 2022_NARA_with grid valu,158820,15,15,0,[5 degree grid]
8,2022 Log Data.xlsx,values with issues,28782,13,13,0,[]
9,2022 Log Data.xlsx,summery,33,2,2,0,[]



2017 Log Data.xlsx :: Sheet1
['LB NUMBER', 'GEAR', '<blank_3>', '<blank_4>', 'LONGI', 'LATTI', '<blank_7>', '<blank_8>', '<blank_9>', '<blank_10>', '<blank_11>', '<blank_12>', '<blank_13>', '<blank_14>', '<blank_15>', 'BAIT', '<blank_17>', '<blank_18>', 'AREA', '<blank_20>', '<blank_21>', 'CATCH', 'KG', 'QTY', 'REG NUM', 'ARRIVAL DATE', 'DEPARTURE DATE', 'ARRIVAL PORT', 'DEPARTURE PORT']

2018 Log data.xlsx :: 2018 Log data
['id', 'Rec_No', 'Y', 'M', 'D', 'Lat', 'Long', 'Fish_Name', 'KG', 'Amount', 'Boat_No', 'Departure_Date', 'Departure_Port', 'Arrival_date', 'Arrival_Port', 'Catch_Type', 'Month', 'Date_Of_Set', 'StartTime', 'EndTime', 'Latitude', 'Longetude', 'Total_Length_of_Net', 'Mesh_Size', 'Depth_Net_Set', 'Height_of_net', 'FADType', 'GillNetLen', 'BranchLine_Len', 'Hooks_Between', 'Total_Hooks', 'Type_of_Bait', 'Remarks', 'Rec_Date', 'EndLatitude', 'EndLongetude', 'zone', 'Fivedegree', 'Onedegree']

2019 Log data.xlsx :: Sheet1
['Rec_No', 'Boat_No', 'Fish_Name', 'KG', 'Amount'

## 4. Schema comparison and candidate roles

In [9]:
def normalise(name: str) -> str:
    return re.sub(r'[^a-z0-9]+', '_', name.lower().strip()).strip('_')
data_headers = {k: v for k, v in headers_by_sheet.items() if v and not re.search(r'summ?e?ry|summary', k, re.I)}
all_fields = sorted({normalise(field) for fields in data_headers.values() for field in fields})
schema_matrix = pd.DataFrame(index=all_fields)
for key, fields in data_headers.items():
    present = {normalise(field) for field in fields}
    schema_matrix[key] = [field in present for field in all_fields]
display(schema_matrix)
patterns = {'time': r'date|time|year|month|^y$|^m$|^d$', 'space': r'lat|lon|long|grid|zone|area|port',
            'species': r'fish|species|spp|^name$', 'catch': r'catch|weight|^kg$|qty|amount',
            'effort': r'hook|net|effort|trip|gear|bait|depth|mesh'}
roles = [{'sheet': key, 'candidate_role': role,
          'observed_fields': [f for f in fields if re.search(pattern, normalise(f))]}
         for key, fields in data_headers.items() for role, pattern in patterns.items()]
candidate_roles = pd.DataFrame(roles)
display(candidate_roles)


,2017 Log Data.xlsx :: Sheet1,2018 Log data.xlsx :: 2018 Log data,2019 Log data.xlsx :: Sheet1,2020 Log Data.xlsx :: Sheet5,2021 LOGBOOK data final.xlsx :: trip 2021,2022 Log Data.xlsx :: catch 2022_NARA_with grid valu,2022 Log Data.xlsx :: values with issues,2023 Log Data.xlsx :: to NARA 2024_v1,2024 Log Data.xlsx :: Sheet1
05_degree_grid,False,False,False,True,False,False,False,False,False
0_degree_grid,False,False,False,True,False,False,False,False,False
1_degree_grid,False,False,False,False,True,False,False,False,False
5_degree_grid,False,False,False,False,True,True,False,False,False
amount,False,True,True,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...
type_of_bait,False,True,True,False,False,False,False,False,False
vessel_id,False,False,False,True,True,True,True,True,True
weight,False,False,False,True,True,True,True,True,True
y,False,True,False,False,False,False,False,False,False


,sheet,candidate_role,observed_fields
0,2017 Log Data.xlsx :: Sheet1,time,"[ARRIVAL DATE, DEPARTURE DATE]"
1,2017 Log Data.xlsx :: Sheet1,space,"[LONGI, LATTI, AREA, ARRIVAL PORT, DEPARTURE PORT]"
2,2017 Log Data.xlsx :: Sheet1,species,[]
3,2017 Log Data.xlsx :: Sheet1,catch,"[CATCH, KG, QTY]"
4,2017 Log Data.xlsx :: Sheet1,effort,"[GEAR, BAIT]"
5,2018 Log data.xlsx :: 2018 Log data,time,"[Y, M, D, Departure_Date, Arrival_date, Month, Date_Of_Set, StartTime, EndTime, Rec_Date]"
6,2018 Log data.xlsx :: 2018 Log data,space,"[Lat, Long, Departure_Port, Arrival_Port, Latitude, Longetude, EndLatitude, EndLongetude, zone]"
7,2018 Log data.xlsx :: 2018 Log data,species,[Fish_Name]
8,2018 Log data.xlsx :: 2018 Log data,catch,"[KG, Amount, Catch_Type]"
9,2018 Log data.xlsx :: 2018 Log data,effort,"[Total_Length_of_Net, Mesh_Size, Depth_Net_Set, Height_of_net, GillNetLen, Hooks_Between, Total_..."


## 5. Bounded fisheries quality sample

This identifies immediate issues without claiming full-dataset statistics. Summary/pivot and empty sheets are excluded. Values are measured but never changed.

In [10]:
def unique_headers(values):
    counts, output = {}, []
    for i, value in enumerate(values, 1):
        base = clean_header(value, i); counts[base] = counts.get(base, 0) + 1
        output.append(base if counts[base] == 1 else f'{base}__{counts[base]}')
    return output

def sample_sheet(path: Path, sheet_name: str, limit: int) -> pd.DataFrame:
    workbook = load_workbook(path, read_only=True, data_only=True)
    try:
        rows = workbook[sheet_name].iter_rows(values_only=True); header = None
        for row in rows:
            if any(v is not None for v in row): header = unique_headers(list(row)); break
        if header is None: return pd.DataFrame()
        data = []
        for row in rows:
            if any(v is not None for v in row): data.append(list(row[:len(header)]))
            if len(data) >= limit: break
        return pd.DataFrame(data, columns=header)
    finally: workbook.close()

quality, sample_frames = [], {}
for record in structure_records:
    if re.search(r'summ?e?ry|summary', record['sheet_name'], re.I) or record['reported_rows'] <= 1: continue
    path = FISHERIES_DIR / record['file_name']; key = f"{path.name} :: {record['sheet_name']}"
    frame = sample_sheet(path, record['sheet_name'], FISHERIES_SAMPLE_ROWS); sample_frames[key] = frame
    placeholder_count = padded_count = 0
    for column in frame.select_dtypes(include='object').columns:
        values = frame[column].dropna().astype(str)
        placeholder_count += int(values.str.strip().str.lower().isin(NULL_TOKENS).sum())
        padded_count += int((values != values.str.strip()).sum())
    quality.append({'sheet': key, 'sample_rows': len(frame), 'sample_columns': len(frame.columns),
        'empty_cells': int(frame.isna().sum().sum()), 'placeholder_text_cells': placeholder_count,
        'padded_text_cells': padded_count, 'duplicate_rows_in_sample': int(frame.duplicated().sum())})
fisheries_sample_quality = pd.DataFrame(quality)
display(fisheries_sample_quality)


,sheet,sample_rows,sample_columns,empty_cells,placeholder_text_cells,padded_text_cells,duplicate_rows_in_sample
0,2017 Log Data.xlsx :: Sheet1,10000,29,0,41569,96784,589
1,2018 Log data.xlsx :: 2018 Log data,10000,39,0,58861,40010,107
2,2019 Log data.xlsx :: Sheet1,10000,35,59691,35540,10071,0
3,2020 Log Data.xlsx :: Sheet5,10000,15,660,0,814,2686
4,2021 LOGBOOK data final.xlsx :: trip 2021,10000,41,598,80339,14,74
5,2022 Log Data.xlsx :: catch 2022_NARA_with grid valu,10000,15,0,0,2252,1116
6,2022 Log Data.xlsx :: values with issues,10000,13,0,0,1041,4942
7,2023 Log Data.xlsx :: to NARA 2024_v1,10000,14,222,798,381,1567
8,2024 Log Data.xlsx :: Sheet1,10000,14,175,0,155,0


## 6. Candidate numeric and date ranges from samples

In [11]:
ranges = []
for key, frame in sample_frames.items():
    for column in frame.columns:
        if not re.search(r'date|time|lat|lon|long|weight|^kg$|qty|amount|hook|grid', normalise(column)): continue
        series = frame[column]; numeric = pd.to_numeric(series, errors='coerce')
        is_date_field = bool(re.search(r'date|time', normalise(column)))
        dates = pd.to_datetime(series, errors='coerce', dayfirst=True) if is_date_field else pd.Series(dtype='datetime64[ns]')
        ranges.append({'sheet': key, 'field': column, 'non_missing': int(series.notna().sum()),
            'numeric_count': int(numeric.notna().sum()), 'numeric_min': numeric.min(), 'numeric_max': numeric.max(),
            'date_count': int(dates.notna().sum()), 'date_min': dates.min(), 'date_max': dates.max()})
candidate_ranges = pd.DataFrame(ranges)
display(candidate_ranges)


,sheet,field,non_missing,numeric_count,numeric_min,numeric_max,date_count,date_min,date_max
0,2017 Log Data.xlsx :: Sheet1,LONGI,10000,0,NaN,NaN,0,NaT,NaT
1,2017 Log Data.xlsx :: Sheet1,LATTI,10000,0,NaN,NaN,0,NaT,NaT
2,2017 Log Data.xlsx :: Sheet1,KG,10000,10000,0.000000e+00,6.000000e+03,0,NaT,NaT
3,2017 Log Data.xlsx :: Sheet1,QTY,10000,10000,0.000000e+00,2.000000e+03,0,NaT,NaT
4,2017 Log Data.xlsx :: Sheet1,ARRIVAL DATE,10000,0,NaN,NaN,9878,2007-11-23,2107-04-10
...,...,...,...,...,...,...,...,...,...
72,2023 Log Data.xlsx :: to NARA 2024_v1,Five Degree Grid Code,10000,10000,6.100050e+06,6.250045e+06,0,NaT,NaT
73,2024 Log Data.xlsx :: Sheet1,qty,10000,10000,0.000000e+00,6.950000e+03,0,NaT,NaT
74,2024 Log Data.xlsx :: Sheet1,weight,10000,10000,0.000000e+00,1.000500e+04,0,NaT,NaT
75,2024 Log Data.xlsx :: Sheet1,date_time,10000,10000,1.704067e+18,1.735603e+18,10000,2024-01-01,2024-12-31


## 7. NetCDF metadata and coverage

Packed variables remain unchanged. Scale factors, offsets, and fill values are reported for correct later decoding.

In [12]:
def attr(dataset, name):
    value = dataset.attrs.get(name)
    if isinstance(value, bytes): return value.decode('utf-8', errors='replace')
    if isinstance(value, np.ndarray): return value.tolist()
    return value.item() if isinstance(value, np.generic) else value

def decode_time(dataset):
    units = str(attr(dataset, 'units') or '')
    if not dataset.size or ' since ' not in units: return None, None
    unit, origin_text = units.split(' since ', 1); origin = datetime.fromisoformat(origin_text.strip())
    factor = 24 if unit.lower().startswith('day') else 1
    return origin + timedelta(hours=float(dataset[0]) * factor), origin + timedelta(hours=float(dataset[-1]) * factor)

def inspect_netcdf(path: Path):
    variables, coverage = [], {'file_name': path.name}
    with h5py.File(path, 'r') as handle:
        for name, ds in handle.items():
            if not isinstance(ds, h5py.Dataset): continue
            variables.append({'file_name': path.name, 'variable': name, 'shape': ds.shape, 'dtype': str(ds.dtype),
                'units': attr(ds, 'units'), 'standard_name': attr(ds, 'standard_name'), 'long_name': attr(ds, 'long_name'),
                'fill_value': attr(ds, '_FillValue'), 'missing_value': attr(ds, 'missing_value'),
                'scale_factor': attr(ds, 'scale_factor'), 'add_offset': attr(ds, 'add_offset')})
            lower = name.lower()
            if lower in {'latitude', 'lat', 'longitude', 'lon', 'depth'} and ds.size:
                values = ds[...]
                coverage.update({f'{lower}_min': float(np.nanmin(values)), f'{lower}_max': float(np.nanmax(values)), f'{lower}_count': int(ds.size)})
            elif lower == 'time':
                start, end = decode_time(ds)
                coverage.update({'time_start': start, 'time_end': end, 'time_count': int(ds.size), 'time_units': attr(ds, 'units')})
    return pd.DataFrame(variables), coverage

variable_frames, coverage_records = [], []
for path in [*chl_files, *sst_files]:
    frame, coverage = inspect_netcdf(path); variable_frames.append(frame); coverage_records.append(coverage)
netcdf_variables = pd.concat(variable_frames, ignore_index=True)
netcdf_coverage = pd.DataFrame(coverage_records)
display(netcdf_variables); display(netcdf_coverage)


,file_name,variable,shape,dtype,units,standard_name,long_name,fill_value,missing_value,scale_factor,add_offset
0,Chla_Western_Southern_SriLanka_daily.nc,CHL,"(10346, 30, 42)",float32,milligram m-3,mass_concentration_of_chlorophyll_a_in_sea_water,Chlorophyll-a concentration - Mean of the binned pixels,[-999.0],[-999.0],None,None
1,Chla_Western_Southern_SriLanka_daily.nc,latitude,"(30,)",float32,degrees_north,latitude,latitude,None,None,None,None
2,Chla_Western_Southern_SriLanka_daily.nc,longitude,"(42,)",float32,degrees_east,longitude,longitude,None,None,None,None
3,Chla_Western_Southern_SriLanka_daily.nc,time,"(10346,)",float32,days since 1900-01-01,time,Time,None,None,None,None
4,SST_SSS_Western_Southern_SriLanka_daily.nc,depth,"(1,)",float32,m,depth,Depth,None,None,None,None
5,SST_SSS_Western_Southern_SriLanka_daily.nc,latitude,"(16,)",float32,degrees_north,latitude,Latitude,None,None,None,None
6,SST_SSS_Western_Southern_SriLanka_daily.nc,longitude,"(22,)",float32,degrees_east,longitude,Longitude,None,None,None,None
7,SST_SSS_Western_Southern_SriLanka_daily.nc,time,"(10346,)",float32,hours since 1950-01-01,time,Time,None,None,None,None
8,SST_SSS_Western_Southern_SriLanka_daily.nc,thetao,"(10346, 1, 16, 22)",int16,degrees_C,sea_water_potential_temperature,Temperature,[-32767],None,[0.0007324442267417908],[21.0]
9,SST_SSS_Western_Southern_SriLanka_daily.nc,so,"(10346, 1, 16, 22)",int16,1e-3,sea_water_salinity,Salinity,[-32767],None,[0.0015259254723787308],[-0.0015259254723787308]


,file_name,latitude_min,latitude_max,latitude_count,longitude_min,longitude_max,longitude_count,time_start,time_end,time_count,time_units,depth_min,depth_max,depth_count
0,Chla_Western_Southern_SriLanka_daily.nc,5.770831,6.979164,30,79.270844,80.979172,42,1997-09-04,2025-12-31,10346,days since 1900-01-01,NaN,NaN,NaN
1,SST_SSS_Western_Southern_SriLanka_daily.nc,5.750000,7.000000,16,79.250000,81.000000,22,1997-09-04,2025-12-31,10346,hours since 1950-01-01,0.494025,0.494025,1.0


## 8. Environmental grid compatibility

In [13]:
def grid_step(lo, hi, count): return (hi - lo) / (count - 1) if count and count > 1 else np.nan
summary = netcdf_coverage.copy()
summary['latitude_step'] = summary.apply(lambda r: grid_step(r.latitude_min, r.latitude_max, r.latitude_count), axis=1)
summary['longitude_step'] = summary.apply(lambda r: grid_step(r.longitude_min, r.longitude_max, r.longitude_count), axis=1)
display(summary)
print('Environmental time overlap:', summary.time_start.max(), 'to', summary.time_end.min())
print('Latitude overlap:', (summary.latitude_min.max(), summary.latitude_max.min()))
print('Longitude overlap:', (summary.longitude_min.max(), summary.longitude_max.min()))
print('The grids overlap but have different resolutions; direct cell-by-cell merging is not justified.')


,file_name,latitude_min,latitude_max,latitude_count,longitude_min,longitude_max,longitude_count,time_start,time_end,time_count,time_units,depth_min,depth_max,depth_count,latitude_step,longitude_step
0,Chla_Western_Southern_SriLanka_daily.nc,5.770831,6.979164,30,79.270844,80.979172,42,1997-09-04,2025-12-31,10346,days since 1900-01-01,NaN,NaN,NaN,0.041667,0.041667
1,SST_SSS_Western_Southern_SriLanka_daily.nc,5.750000,7.000000,16,79.250000,81.000000,22,1997-09-04,2025-12-31,10346,hours since 1950-01-01,0.494025,0.494025,1.0,0.083333,0.083333


Environmental time overlap: 1997-09-04 00:00:00 to 2025-12-31 00:00:00
Latitude overlap: (5.7708306312561035, 6.979164123535156)
Longitude overlap: (79.27084350585938, 80.97917175292969)
The grids overlap but have different resolutions; direct cell-by-cell merging is not justified.


## 9. Save audit evidence

Only audit summaries are written to `outputs/audit`. Raw files are never changed or copied. Re-running the notebook deliberately replaces the previous audit tables with results from the current files and settings.

In [14]:
audit_tables = {
    'dataset_inventory.csv': inventory,
    'fisheries_workbook_structure.csv': fisheries_structure,
    'fisheries_schema_matrix.csv': schema_matrix.reset_index(names='normalised_field'),
    'fisheries_candidate_roles.csv': candidate_roles,
    'fisheries_sample_quality.csv': fisheries_sample_quality,
    'fisheries_candidate_ranges.csv': candidate_ranges,
    'netcdf_variables.csv': netcdf_variables,
    'netcdf_coverage.csv': netcdf_coverage,
    'environmental_grid_summary.csv': summary,
}
for file_name, table in audit_tables.items():
    table.to_csv(AUDIT_OUTPUT_DIR / file_name, index=False)

manifest = {
    'created_utc': pd.Timestamp.now(tz='UTC').isoformat(),
    'runtime': 'Google Colab' if RUNNING_IN_COLAB else 'Local Windows',
    'raw_root': str(RAW_ROOT),
    'fisheries_sample_rows_per_sheet': FISHERIES_SAMPLE_ROWS,
    'input_files': inventory[['relative_path', 'size_mb']].to_dict(orient='records'),
    'output_files': sorted(audit_tables),
}
with (AUDIT_OUTPUT_DIR / 'audit_manifest.json').open('w', encoding='utf-8') as handle:
    json.dump(manifest, handle, indent=2, ensure_ascii=False)

print(f'Saved {len(audit_tables)} audit tables and audit_manifest.json to:')
print(AUDIT_OUTPUT_DIR)


Saved 9 audit tables and audit_manifest.json to:
/content/drive/MyDrive/FishGuard-C3/outputs/audit


## 10. Initial findings

- The fisheries source contains eight annual Excel workbooks (2017–2024), not CSV files.
- Fisheries schemas change materially across years. Field names, dates, coordinates, and effort-related fields are inconsistent.
- Some workbooks contain summaries, pivots, empty sheets, or sheets explicitly identifying problematic values. These must not be combined blindly.
- Candidate catch fields exist (`KG`, `weight`, `qty`, and `Amount`), but their definitions and units still require verification.
- Candidate effort fields exist in some years, but availability and comparability are inconsistent. **CPUE feasibility is not established.**
- Coordinates use direction-prefixed text, decimal values, direction fields, and grid codes. Coordinate validity must be assessed before matching.
- Chlorophyll-a uses `CHL` in `milligram m-3`, with `-999` as its missing-value convention, on a 30 × 42 daily grid.
- SST is packed in `thetao` and must be decoded with its scale factor, offset, and fill value. The file also contains salinity (`so`), which is not an approved input at this stage.
- Both environmental datasets span 1997-09-04 to 2025-12-31. Their spatial areas overlap, but grid resolutions differ.
- Temporal aggregation and spatial matching must wait until trustworthy fisheries dates and coordinates are established.

## 11. Questions before preprocessing

1. Which sheet in each workbook is the authoritative row-level catch table?
2. What does one row represent: species within a set, species within a trip, a landing record, or another unit?
3. Are `KG` and `weight` consistently kilograms, and what do `qty` and `Amount` represent?
4. Which date represents the fishing event when departure, arrival, set, and record dates coexist?
5. Which coordinate pair or grid field is authoritative for each year?
6. Is any effort measure sufficiently complete and comparable across years to define CPUE?
7. How should records marked as problematic be resolved and documented?
8. Does the trustworthy fisheries coverage overlap the environmental subset?

**Recommended next step:** confirm authoritative sheets and field meanings with the data provider, then prepare a documented schema mapping and full streaming quality audit. Do not clean, merge, calculate CPUE, or train models yet.